In [1]:
import pandas as pd
import numpy as np
import yaml

from sklearn.model_selection import train_test_split
from pycaret.classification import *

import mlflow


In [2]:
# Muat konfigurasi model
with open('../../configs/model_config.yaml', 'r') as file:
    config = yaml.safe_load(file)

print(f"Target: {config['target_column']}")
print(f"Test Size: {config['test_size']}")
print(f"Random Seed: {config['random_seed']}")


Target: operation_type
Test Size: 0.2
Random Seed: 42


In [3]:
path_raw = "../data/processed/hrss_clean_integrated_original_columns.csv"
path_eng = "../data/processed/hrss_clean_integrated_with_engineered_features.csv"

df_raw = pd.read_csv(path_raw)
df_eng = pd.read_csv(path_eng)

# Drop leakage column (operation_name is a text representation of target operation_type)
if 'operation_name' in df_raw.columns:
    df_raw = df_raw.drop(columns=['operation_name'])
if 'operation_name' in df_eng.columns:
    df_eng = df_eng.drop(columns=['operation_name'])

print(df_raw.shape)
print(df_eng.shape)

(46898, 20)
(46898, 26)


In [4]:
print("RAW DATA INFO")
display(df_raw.info())

print("\nENGINEERED DATA INFO")
display(df_eng.info())

RAW DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46898 entries, 0 to 46897
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Timestamp        46898 non-null  float64
 1   I_w_BLO_Weg      46898 non-null  int64  
 2   O_w_BLO_power    46898 non-null  int64  
 3   O_w_BLO_voltage  46898 non-null  int64  
 4   I_w_BHL_Weg      46898 non-null  int64  
 5   O_w_BHL_power    46898 non-null  int64  
 6   O_w_BHL_voltage  46898 non-null  int64  
 7   I_w_BHR_Weg      46898 non-null  int64  
 8   O_w_BHR_power    46898 non-null  int64  
 9   O_w_BHR_voltage  46898 non-null  int64  
 10  I_w_BRU_Weg      46898 non-null  int64  
 11  O_w_BRU_power    46898 non-null  int64  
 12  O_w_BRU_voltage  46898 non-null  int64  
 13  I_w_HR_Weg       46898 non-null  int64  
 14  O_w_HR_power     46898 non-null  int64  
 15  O_w_HR_voltage   46898 non-null  int64  
 16  I_w_HL_Weg       46898 non-null  int64  
 17

None


ENGINEERED DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46898 entries, 0 to 46897
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Timestamp               46898 non-null  float64
 1   I_w_BLO_Weg             46898 non-null  int64  
 2   O_w_BLO_power           46898 non-null  int64  
 3   O_w_BLO_voltage         46898 non-null  int64  
 4   I_w_BHL_Weg             46898 non-null  int64  
 5   O_w_BHL_power           46898 non-null  int64  
 6   O_w_BHL_voltage         46898 non-null  int64  
 7   I_w_BHR_Weg             46898 non-null  int64  
 8   O_w_BHR_power           46898 non-null  int64  
 9   O_w_BHR_voltage         46898 non-null  int64  
 10  I_w_BRU_Weg             46898 non-null  int64  
 11  O_w_BRU_power           46898 non-null  int64  
 12  O_w_BRU_voltage         46898 non-null  int64  
 13  I_w_HR_Weg              46898 non-null  int64  
 14  O_w_HR_power    

None

In [5]:
print("RAW TARGET DISTRIBUTION")
print(df_raw['operation_type'].value_counts(normalize=True))

print("\nENGINEERED TARGET DISTRIBUTION")
print(df_eng['operation_type'].value_counts(normalize=True))

RAW TARGET DISTRIBUTION
operation_type
0    0.54851
1    0.45149
Name: proportion, dtype: float64

ENGINEERED TARGET DISTRIBUTION
operation_type
0    0.54851
1    0.45149
Name: proportion, dtype: float64


In [6]:
target = "operation_type"

X_raw = df_raw.drop(columns=[target])
y_raw = df_raw[target]

X_eng = df_eng.drop(columns=[target])
y_eng = df_eng[target]

print(X_raw.shape, y_raw.shape)
print(X_eng.shape, y_eng.shape)

(46898, 19) (46898,)
(46898, 25) (46898,)


In [7]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=config['test_size'],
    random_state=config['random_seed'],
    stratify=y_raw
)

X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_eng, y_eng,
    test_size=config['test_size'],
    random_state=config['random_seed'],
    stratify=y_eng
)

print(X_train_raw.shape, X_test_raw.shape)
print(X_train_eng.shape, X_test_eng.shape)


(37518, 19) (9380, 19)
(37518, 25) (9380, 25)


In [8]:
import os

os.makedirs("../data/splits", exist_ok=True)

# RAW
X_train_raw.to_csv("../data/splits/X_train_raw.csv", index=False)
X_test_raw.to_csv("../data/splits/X_test_raw.csv", index=False)
y_train_raw.to_csv("../data/splits/y_train_raw.csv", index=False)
y_test_raw.to_csv("../data/splits/y_test_raw.csv", index=False)

# ENGINEERED
X_train_eng.to_csv("../data/splits/X_train_eng.csv", index=False)
X_test_eng.to_csv("../data/splits/X_test_eng.csv", index=False)
y_train_eng.to_csv("../data/splits/y_train_eng.csv", index=False)
y_test_eng.to_csv("../data/splits/y_test_eng.csv", index=False)

In [9]:
# ============================================================
# LOAD SPLITS FROM DISK (data/splits/)
# ============================================================
print("Memuat data split dari data/splits/...")

X_train_raw = pd.read_csv("../data/splits/X_train_raw.csv")
X_test_raw = pd.read_csv("../data/splits/X_test_raw.csv")
y_train_raw = pd.read_csv("../data/splits/y_train_raw.csv")
y_test_raw = pd.read_csv("../data/splits/y_test_raw.csv")

X_train_eng = pd.read_csv("../data/splits/X_train_eng.csv")
X_test_eng = pd.read_csv("../data/splits/X_test_eng.csv")
y_train_eng = pd.read_csv("../data/splits/y_train_eng.csv")
y_test_eng = pd.read_csv("../data/splits/y_test_eng.csv")

# Siapkan data training dan testing untuk PyCaret
train_data_raw = pd.concat([X_train_raw, y_train_raw], axis=1)
test_data_raw = pd.concat([X_test_raw, y_test_raw], axis=1)

train_data_eng = pd.concat([X_train_eng, y_train_eng], axis=1)
test_data_eng = pd.concat([X_test_eng, y_test_eng], axis=1)

print(f"RAW Train shape: {train_data_raw.shape}, Test shape: {test_data_raw.shape}")
print(f"ENGINEERED Train shape: {train_data_eng.shape}, Test shape: {test_data_eng.shape}")


Memuat data split dari data/splits/...
RAW Train shape: (37518, 20), Test shape: (9380, 20)
ENGINEERED Train shape: (37518, 26), Test shape: (9380, 26)


In [11]:
mlflow.set_tracking_uri("file:../mlruns")

mlflow.set_experiment("HRSS_Operational_Classification")

<Experiment: artifact_location='file:///d:/ARFI/Kuliah/Project/industrial_recommendation_system/notebooks/../mlruns/691691579525109979', creation_time=1779957094958, experiment_id='691691579525109979', last_update_time=1779957094958, lifecycle_stage='active', name='HRSS_Operational_Classification', tags={}>

In [12]:
experiment_config = {
    "target": "operation_type",
    "problem_type": "binary_classification",
    "domain": "industrial_telemetry_hrss",
    "objective": "operational_efficiency_classification",
    "primary_metric": "f1_macro",
    "secondary_metric": "recall_class_1"
}

print(experiment_config)

{'target': 'operation_type', 'problem_type': 'binary_classification', 'domain': 'industrial_telemetry_hrss', 'objective': 'operational_efficiency_classification', 'primary_metric': 'f1_macro', 'secondary_metric': 'recall_class_1'}


In [13]:
experiment_registry = {
    "raw": {
        "description": "Baseline model using original telemetry features only",
        "dataset_shape": X_train_raw.shape
    },
    "engineered": {
        "description": "Enhanced model using engineered operational features",
        "dataset_shape": X_train_eng.shape
    }
}

print(experiment_registry)

{'raw': {'description': 'Baseline model using original telemetry features only', 'dataset_shape': (37518, 19)}, 'engineered': {'description': 'Enhanced model using engineered operational features', 'dataset_shape': (37518, 25)}}


In [14]:
print("READY FOR EXPERIMENTS")

print("\nRAW:")
print(X_train_raw.shape, X_test_raw.shape)

print("\nENGINEERED:")
print(X_train_eng.shape, X_test_eng.shape)

READY FOR EXPERIMENTS

RAW:
(37518, 19) (9380, 19)

ENGINEERED:
(37518, 25) (9380, 25)


### Raw Data

In [24]:
# ============================================================
# PYCARET EXPERIMENT - RAW DATA
# ============================================================
print("Memulai PyCaret Experiment untuk RAW DATA...")

exp_raw = ClassificationExperiment()
exp_raw.setup(
    data=train_data_raw,
    test_data=test_data_raw,
    target=config['target_column'],
    session_id=config['random_seed'],
    experiment_name="HRSS_RAW_Baseline",
    log_experiment=True,
    log_plots=True,
    index=False
)



Memulai PyCaret Experiment untuk RAW DATA...


,Description,Value
0,Session id,42
1,Target,operation_type
2,Target type,Binary
3,Original data shape,"(46898, 20)"
4,Transformed data shape,"(46898, 20)"
5,Transformed train set shape,"(37518, 20)"
6,Transformed test set shape,"(9380, 20)"
7,Numeric features,19
8,Preprocess,True
9,Imputation type,simple


In [17]:
# Membandingkan model
best_model_raw = exp_raw.compare_models(fold=config['pycaret']['fold'], sort=config['pycaret']['sort_metric'])


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.5100
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.3100
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.4100
ada,Ada Boost Classifier,0.9999,0.9999,0.9999,1.0000,0.9999,0.9999,0.9999,0.5160
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9998,0.9999,0.9997,0.9997,0.1420
gbc,Gradient Boosting Classifier,0.9998,1.0000,0.9996,0.9999,0.9998,0.9996,0.9996,1.7040
lightgbm,Light Gradient Boosting Machine,0.9998,0.9999,0.9997,0.9999,0.9998,0.9996,0.9996,0.2920
dt,Decision Tree Classifier,0.9995,0.9995,0.9994,0.9995,0.9995,0.9990,0.9990,0.0900
qda,Quadratic Discriminant Analysis,0.9523,0.9914,0.9155,0.9774,0.9454,0.9031,0.9047,0.0720
knn,K Neighbors Classifier,0.9499,0.9916,0.9344,0.9538,0.9440,0.8988,0.8989,0.6420


In [18]:
results_raw = exp_raw.pull()
display(results_raw)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.510
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.310
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.410
ada,Ada Boost Classifier,0.9999,0.9999,0.9999,1.0000,0.9999,0.9999,0.9999,0.516
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9998,0.9999,0.9997,0.9997,0.142
gbc,Gradient Boosting Classifier,0.9998,1.0000,0.9996,0.9999,0.9998,0.9996,0.9996,1.704
lightgbm,Light Gradient Boosting Machine,0.9998,0.9999,0.9997,0.9999,0.9998,0.9996,0.9996,0.292
dt,Decision Tree Classifier,0.9995,0.9995,0.9994,0.9995,0.9995,0.9990,0.9990,0.090
qda,Quadratic Discriminant Analysis,0.9523,0.9914,0.9155,0.9774,0.9454,0.9031,0.9047,0.072
knn,K Neighbors Classifier,0.9499,0.9916,0.9344,0.9538,0.9440,0.8988,0.8989,0.642


### Engineered Data

In [19]:
# ============================================================
# PYCARET EXPERIMENT - ENGINEERED DATA
# ============================================================
print("Memulai PyCaret Experiment untuk ENGINEERED DATA...")

exp_eng = ClassificationExperiment()
exp_eng.setup(
    data=train_data_eng,
    test_data=test_data_eng,
    target=config['target_column'],
    session_id=config['random_seed'],
    experiment_name="HRSS_ENGINEERED_Features",
    log_experiment=True,
    log_plots=True,
    index=False
)

# Membandingkan model
best_model_eng = exp_eng.compare_models(fold=config['pycaret']['fold'], sort=config['pycaret']['sort_metric'])

results_eng = exp_eng.pull()
display(results_eng)


Memulai PyCaret Experiment untuk ENGINEERED DATA...


,Description,Value
0,Session id,42
1,Target,operation_type
2,Target type,Binary
3,Original data shape,"(46898, 26)"
4,Transformed data shape,"(46898, 26)"
5,Transformed train set shape,"(37518, 26)"
6,Transformed test set shape,"(9380, 26)"
7,Numeric features,25
8,Preprocess,True
9,Imputation type,simple


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.5620
ada,Ada Boost Classifier,1.0000,0.9999,0.9999,1.0000,1.0000,0.9999,0.9999,0.6200
gbc,Gradient Boosting Classifier,1.0000,1.0000,0.9999,1.0000,1.0000,0.9999,0.9999,2.2720
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.3320
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.1960
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9999,0.9999,0.9998,0.9998,0.1560
lightgbm,Light Gradient Boosting Machine,0.9997,0.9999,0.9996,0.9998,0.9997,0.9994,0.9994,0.2760
dt,Decision Tree Classifier,0.9995,0.9996,0.9996,0.9994,0.9995,0.9991,0.9991,0.1100
knn,K Neighbors Classifier,0.9512,0.9921,0.9390,0.9523,0.9456,0.9014,0.9015,0.6200
qda,Quadratic Discriminant Analysis,0.9234,0.9730,0.9754,0.8762,0.9215,0.8475,0.8551,0.0720


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.562
ada,Ada Boost Classifier,1.0000,0.9999,0.9999,1.0000,1.0000,0.9999,0.9999,0.620
gbc,Gradient Boosting Classifier,1.0000,1.0000,0.9999,1.0000,1.0000,0.9999,0.9999,2.272
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.332
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.196
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9999,0.9999,0.9998,0.9998,0.156
lightgbm,Light Gradient Boosting Machine,0.9997,0.9999,0.9996,0.9998,0.9997,0.9994,0.9994,0.276
dt,Decision Tree Classifier,0.9995,0.9996,0.9996,0.9994,0.9995,0.9991,0.9991,0.110
knn,K Neighbors Classifier,0.9512,0.9921,0.9390,0.9523,0.9456,0.9014,0.9015,0.620
qda,Quadratic Discriminant Analysis,0.9234,0.9730,0.9754,0.8762,0.9215,0.8475,0.8551,0.072


In [20]:
# Membandingkan model
best_model_eng = exp_eng.compare_models(fold=config['pycaret']['fold'], sort=config['pycaret']['sort_metric'])



,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.5500
ada,Ada Boost Classifier,1.0000,0.9999,0.9999,1.0000,1.0000,0.9999,0.9999,0.6160
gbc,Gradient Boosting Classifier,1.0000,1.0000,0.9999,1.0000,1.0000,0.9999,0.9999,2.2600
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.3260
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.2260
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9999,0.9999,0.9998,0.9998,0.1540
lightgbm,Light Gradient Boosting Machine,0.9997,0.9999,0.9996,0.9998,0.9997,0.9994,0.9994,0.2760
dt,Decision Tree Classifier,0.9995,0.9996,0.9996,0.9994,0.9995,0.9991,0.9991,0.1040
knn,K Neighbors Classifier,0.9512,0.9921,0.9390,0.9523,0.9456,0.9014,0.9015,0.6200
qda,Quadratic Discriminant Analysis,0.9234,0.9730,0.9754,0.8762,0.9215,0.8475,0.8551,0.0680


In [21]:
results_eng = exp_eng.pull()
display(results_eng)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.550
ada,Ada Boost Classifier,1.0000,0.9999,0.9999,1.0000,1.0000,0.9999,0.9999,0.616
gbc,Gradient Boosting Classifier,1.0000,1.0000,0.9999,1.0000,1.0000,0.9999,0.9999,2.260
et,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.326
catboost,CatBoost Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,6.226
xgboost,Extreme Gradient Boosting,0.9999,1.0000,0.9999,0.9999,0.9999,0.9998,0.9998,0.154
lightgbm,Light Gradient Boosting Machine,0.9997,0.9999,0.9996,0.9998,0.9997,0.9994,0.9994,0.276
dt,Decision Tree Classifier,0.9995,0.9996,0.9996,0.9994,0.9995,0.9991,0.9991,0.104
knn,K Neighbors Classifier,0.9512,0.9921,0.9390,0.9523,0.9456,0.9014,0.9015,0.620
qda,Quadratic Discriminant Analysis,0.9234,0.9730,0.9754,0.8762,0.9215,0.8475,0.8551,0.068


In [ ]:
# ============================================================
# PREDICT & EVALUATE - RAW DATA
# ============================================================
print("Evaluasi Model Terbaik (RAW DATA)...")

# Gabungkan X_test dan y_test untuk digunakan di predict_model
test_data_raw = pd.concat([X_test_raw, y_test_raw], axis=1)

# Prediksi pada data testing unseen
predictions_raw = exp_raw.predict_model(best_model_raw, data=test_data_raw)
display(predictions_raw.head())

# Tampilkan UI Interaktif Evaluasi PyCaret
exp_raw.evaluate_model(best_model_raw)


Evaluasi Model Terbaik (RAW DATA)...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


,Timestamp,I_w_BLO_Weg,O_w_BLO_power,O_w_BLO_voltage,I_w_BHL_Weg,O_w_BHL_power,O_w_BHL_voltage,I_w_BHR_Weg,O_w_BHR_power,O_w_BHR_voltage,...,O_w_BRU_voltage,I_w_HR_Weg,O_w_HR_power,O_w_HR_voltage,I_w_HL_Weg,O_w_HL_power,O_w_HL_voltage,operation_type,prediction_label,prediction_score
0,14.828995,8,0,0,-547,0,0,-873,71,1,...,60,0,5504,26,0,5008,23,0,0,1.0
1,3.736000,8,0,0,480,13876,93,782,0,0,...,0,-817,10264,77,-817,24391,77,1,1,1.0
2,10.546005,-107,0,0,0,0,0,-1272,1968,23,...,54,0,12214,27,0,12966,26,1,1,1.0
3,3.178001,8,0,0,76,21187,99,762,6314,41,...,1,-424,32669,266,-428,11728,266,1,1,1.0
4,4.207001,0,0,0,637,13546,94,783,20,1,...,0,-833,13483,27,-833,22825,31,1,1,1.0


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [23]:
# ============================================================
# PREDICT & EVALUATE - ENGINEERED DATA
# ============================================================
print("Evaluasi Model Terbaik (ENGINEERED DATA)...")

# Gabungkan X_test dan y_test untuk digunakan di predict_model
test_data_eng = pd.concat([X_test_eng, y_test_eng], axis=1)

# Prediksi pada data testing unseen
predictions_eng = exp_eng.predict_model(best_model_eng, data=test_data_eng)
display(predictions_eng.head())

# Tampilkan UI Interaktif Evaluasi PyCaret
exp_eng.evaluate_model(best_model_eng)


Evaluasi Model Terbaik (ENGINEERED DATA)...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


,Timestamp,I_w_BLO_Weg,O_w_BLO_power,O_w_BLO_voltage,I_w_BHL_Weg,O_w_BHL_power,O_w_BHL_voltage,I_w_BHR_Weg,O_w_BHR_power,O_w_BHR_voltage,...,O_w_HL_voltage,total_power,avg_voltage,total_movement,power_efficiency_ratio,rail_activity,conveyor_activity,operation_type,prediction_label,prediction_score
0,14.828995,8,0,0,-547,0,0,-873,71,1,...,23,38183,18.333334,1960,19.471188,0,1960,0,0,1.0
1,3.736000,8,0,0,480,13876,93,782,0,0,...,77,48535,41.166668,3470,13.983002,1634,1836,1,1,1.0
2,10.546005,-107,0,0,0,0,0,-1272,1968,23,...,26,51796,21.666666,1585,32.658260,0,1585,1,1,1.0
3,3.178001,8,0,0,76,21187,99,762,6314,41,...,266,71990,112.166664,2263,31.797703,852,1411,1,1,1.0
4,4.207001,0,0,0,637,13546,94,783,20,1,...,31,49874,25.500000,3651,13.656627,1666,1985,1,1,1.0


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…